In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd

In [13]:
model = joblib.load("models/xgboost.pk1")
scaler = joblib.load("models/scaler.pk1")
label_encoder = joblib.load("models/label_encoder.pk1")

In [15]:
with open("mitigation/mitigation_database.json","r") as f:
    mitigation_db=json.load(f)

In [17]:
def predict_attack(features):
    features=np.array(features).reshape(1,-1)
    scaled=scaler.transform(features)
    prediction=model.predict(scaled)
    attack=label_encoder.inverse_transform(prediction)[0]
    return attack

In [37]:
def predict_attack_with_confidence(features):
    features=np.array(features).reshape(1,-1)
    scaled=scaler.transform(features)
    probabilities=model.predict_proba(scaled)
    confidence=np.max(probabilities)*100
    prediction=model.predict(scaled)
    attack=label_encoder.inverse_transform(prediction)[0]
    return attack,confidence

In [45]:
def detect_and_mitigate(features):       #mitigation Engine
    attack,confidence=predict_attack_with_confidence(features)
    mitigation=mitigation_db.get(attack)
    result={
        "Prediction":attack,
        "Confidence":round(confidence,2),
        "Severity":mitigation["Severity"],
        "Description":mitigation["Description"],
        "Mitigation":mitigation["Mitigation"]
    }
    return result

In [33]:
X_test=pd.read_csv("data/processed/X_test.csv") #load test dataset

In [47]:
sample=X_test.iloc[0].values
result=detect_and_mitigate(sample)  #testing one sample

C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [49]:
print(result)

{'Prediction': 'BENIGN', 'Confidence': 99.84, 'Severity': 'None', 'Description': 'Normal network traffic.', 'Mitigation': ['No action required.']}


In [51]:
print("="*60)
print("Prediction :",result["Prediction"])
print("Confidence :",result["Confidence"],"%")
print("Severity :",result["Severity"])
print()
print("Description")
print(result["Description"])
print()
print("Mitigation")
for step in result["Mitigation"]:
    print("-",step)

Prediction : BENIGN
Confidence : 99.84 %
Severity : None

Description
Normal network traffic.

Mitigation
- No action required.


In [53]:
report = {
    "Prediction":[result["Prediction"]],
    "Confidence":[result["Confidence"]],
    "Severity":[result["Severity"]],
    "Description":[result["Description"]],
    "Mitigation":[" | ".join(result["Mitigation"])]
}
report = pd.DataFrame(report)
report.to_csv(
    "integration/latest_detection.csv",
    index=False
)

In [57]:
results=[]            #multiple prediction
for i in range(20):
    sample = X_test.iloc[i].values
    output = detect_and_mitigate(sample)
    results.append({
        "Prediction":output["Prediction"],
        "Confidence":output["Confidence"],
        "Severity":output["Severity"]
    })

C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\HP\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but 

In [59]:
results = pd.DataFrame(results)
results.head()

,Prediction,Confidence,Severity
0,BENIGN,99.84,None
1,BENIGN,99.98,None
2,BENIGN,99.95,None
3,BENIGN,99.80,None
4,BENIGN,99.87,None


In [61]:
results.to_csv(                               #saving batch record
    "integration/batch_detection.csv",
    index=False
) 

In [63]:
attack_counts = results["Prediction"].value_counts()   #count attack types
print(attack_counts)

Prediction
BENIGN    20
Name: count, dtype: int64


In [65]:
severity_counts = results["Severity"].value_counts()  #severity distribution
print(severity_counts)

Severity
None    20
Name: count, dtype: int64


In [67]:
print("Total Predictions :",len(results))   #prediction statistics
print("Average Confidence :",results["Confidence"].mean())
print("Maximum Confidence :",results["Confidence"].max())
print("Minimum Confidence :",results["Confidence"].min())

Total Predictions : 20
Average Confidence : 99.844
Maximum Confidence : 99.98
Minimum Confidence : 99.12


In [71]:
print("="*60)
print("SYSTEM INTEGRATION SUCCESSFUL")
print("="*60)



SYSTEM INTEGRATION SUCCESSFUL
